Section 1:
is_declining is a handful numeric features and week4 baseline build on one signal so i would get start with logistic regression as primary model due to natural next step up from single signal rule 

Section 2: This one firm right answer given what i found on week 4: client-grouped-split due to:
- Week 4 exposed real client concentration and one client have 3 of top 10 flagged pages.
- In this case if we split randomly, pages from the same client can end up in both train and test and the model could learn client X's page tend to decline rather than a generalize page-level pattern.

In [ ]:
import os
from huggingface_hub import hf_hub_download
import duckdb
import pandas as pd

# download
path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)

con = duckdb.connect()

# Aggregate to page-level: first half vs second half of March
df = con.execute(f"""
    WITH daily AS (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,
            gsc_impressions,
            gsc_avg_position,
            gsc_clicks,
            CASE WHEN report_date <= DATE '2026-03-15' THEN 'first_half' ELSE 'second_half' END AS period
        FROM read_parquet('{path}')
        WHERE gsc_data_available IS TRUE
    ),
    agg AS (
        SELECT
            content_hash_id,
            client_hash_id,
            period,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            AVG(gsc_avg_position) AS avg_position
        FROM daily
        GROUP BY 1,2,3
    )
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        f.impressions AS impressions_first,
        s.impressions AS impressions_second,
        f.avg_position AS avg_position_first,
        s.avg_position AS avg_position_second
    FROM agg f
    JOIN agg s
      ON f.content_hash_id = s.content_hash_id
     AND f.client_hash_id = s.client_hash_id
    WHERE f.period = 'first_half' AND s.period = 'second_half'
""").df()

df.shape

(141467, 6)

In [9]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_hash_id"]))

train = df.iloc[train_idx]
test = df.iloc[test_idx]

print(train.shape, test.shape)
print("Unique clients in train:", train["client_hash_id"].nunique())
print("Unique clients in test:", test["client_hash_id"].nunique())

(131220, 7) (10247, 7)
Unique clients in train: 34
Unique clients in test: 9


In [15]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_hash_id"]))

train = df.iloc[train_idx]
test = df.iloc[test_idx]

print(train.shape, test.shape)
print(train.columns.tolist())

(131220, 9) (10247, 9)
['content_hash_id', 'client_hash_id', 'impressions_first', 'impressions_second', 'avg_position_first', 'avg_position_second', 'position_bucket', 'pct_change_impressions', 'is_declining']


Section 3: Train + compare vs. baseline



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

feature_cols = ["impressions_first", "avg_position_first"]

X_train, y_train = train[feature_cols], train["is_declining"]
X_test, y_test = test[feature_cols], test["is_declining"]

#Logistic Regression 
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
logreg_probs = logreg.predict_proba(X_test)[:, 1]
logreg_auc = roc_auc_score(y_test, logreg_probs)

#Decision Tree
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(X_train, y_train)
tree_probs = tree.predict_proba(X_test)[:, 1]
tree_auc = roc_auc_score(y_test, tree_probs)

#Baseline: rank by impressions_first alone (Week 4 rule)
baseline_auc = roc_auc_score(y_test, X_test["impressions_first"])

print("Baseline:", round(baseline_auc, 4))
print("Logistic Regression:              ", round(logreg_auc, 4))
print("Decision Tree (max_depth=4):       ", round(tree_auc, 4))

Baseline: 0.5448
Logistic Regression:               0.4962
Decision Tree (max_depth=4):        0.5511


In [ ]:
#Trying Standard Scaler to chech whether result improve or not
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logreg_scaled = LogisticRegression(max_iter=1000)
logreg_scaled.fit(X_train_scaled, y_train)
logreg_scaled_auc = roc_auc_score(y_test, logreg_scaled.predict_proba(X_test_scaled)[:, 1])
print("Logistic Regression (scaled):", round(logreg_scaled_auc, 4))

Logistic Regression (scaled): 0.4962


## 3. Train + Compare vs. Baseline

Features: `impressions_first`, `avg_position_first` (both pre-decision-point).

Split: client-grouped (34 train clients, 9 test clients, no overlap).
Metric: ROC AUC (consistent with Week 2's chosen metric).

| Method | AUC |
|---|---|
| Baseline (impressions_first only) | 0.5448 |
| Logistic Regression | 0.4962 |
| Logistic Regression (scaled) | 0.4962 |
| Decision Tree (max_depth=4) | 0.5511 |
